In [59]:
import networkx as nx
import pandas as pd
import numpy as np

In [74]:
pools = pd.read_csv(
    '../block_extractor/data/pools.csv',
    index_col=0,
    names=["index", "address", "version", "token0", "token1", "fee", "block_number", "timestamp", "tickspacing"]
).sort_index().drop_duplicates()
tokens = pd.read_csv(
    '../block_extractor/data/tokens.csv',
    index_col=0,
    names=["index", "address", "name", "symbol", "decimals"]
).sort_index().drop_duplicates()
print(len(pools))
print(len(tokens))

430384
406659


In [75]:
pools.head()

,address,version,token0,token1,fee,block_number,timestamp,tickspacing
index,,,,,,,,
1,0x00001bea43608c5ee487f82b773af8bd7cb20a6f,2,0x12898dabdbedd0fdc5c9ea4448d532ab55d92740,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,17589578,1688097587,0
2,0x000024feb293b6c6c3a80a95f1f830a8746400b9,3,0x6aa56e1d98b3805921c170eb4b3fe7d4fda6d89b,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,10000,20425838,1722420503,200
3,0x00004ee988665cdda9a1080d5792cecd16dc1220,2,0x4d44d6c288b7f32ff676a4b2dafd625992f8ffbd,0xdac17f958d2ee523a2206206994597c13d831ec7,3000,11666388,1610801924,0
4,0x0000871c95bb027c90089f4926fd1ba82cdd9a8b,2,0x5152e1cb69a2ffa3997e89cbb4aba76a01d82141,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,10986295,1601773155,0
5,0x000089906c37426585e860d02c545ab1d184a6ba,2,0x722383e1e8994cb8a55cbc1621dc068b62403b1e,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,19685531,1713481823,0


In [76]:
tokens.head()

,address,name,symbol,decimals
index,,,,
0,0x12898dabdbedd0fdc5c9ea4448d532ab55d92740,Kabosu 2.0,KABOSU2.0,9
1,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,Wrapped Ether,WETH,18
2,0x6aa56e1d98b3805921c170eb4b3fe7d4fda6d89b,MAGA,TRUMP,9
3,0x4d44d6c288b7f32ff676a4b2dafd625992f8ffbd,Solution Life Coin,SLC,18
4,0xdac17f958d2ee523a2206206994597c13d831ec7,Tether USD,USDT,6


In [77]:
old_pools = pools[pools['block_number'] < 14771015]
all_pools = old_pools[['token0', 'token1']].to_numpy().tolist()
G = nx.from_edgelist(all_pools, create_using=nx.MultiDiGraph)
print("all_pools: ", len(all_pools))
print("tokens: ", len(tokens))
print("nodes: ", len(G.nodes))
print("edges: ", len(G.edges))

all_pools:  80572
tokens:  406659
nodes:  71092
edges:  80572


In [78]:
remove = [node for (node, d) in dict(G.degree()).items() if d <= 40]
G.remove_nodes_from(remove)
print("nodes: ", len(G.nodes))
print("edges: ", len(G.edges))

nodes:  25
edges:  326


In [79]:
# filter based on degree
edges = list(G.edges())
mask = []
for (t0, t1) in all_pools:
    if (t0, t1) in edges or (t1, t0) in edges:
        mask.append(True)
    else:
        mask.append(False)
masked_pools= old_pools[mask]
masked_pools.to_csv('../block_extractor/data/filtered_pools.csv', sep=',', header=False)

In [80]:
filtered_tokens = list(G.nodes())
mask = []
for t in tokens['address']:
    if t in filtered_tokens:
        mask.append(True)
    else:
        mask.append(False)
masked_tokens = tokens[mask]
masked_tokens.to_csv('../block_extractor/data/filtered_tokens.csv', sep=',', header=False)

In [81]:
prices = pd.read_parquet('../block_extractor/data/prices.parquet')
prices.reset_index()
prices.head()

,pool_address,block_number,reserve_t0,reserve_t1,sqrt_price_x96
0,0x02dd9ff64eec5d866a512ef08463c7c85a14e4aa,14763568,113059331313379607023,78828382,None
1,0x033fa3759a33b2d3acf50d5166beaa1db53fb2a0,14763568,993704288414882406184,952358335000697494352,None
2,0x0357079bbecadd7bd4b7a9f418014679fc4e3926,14763568,114658834913402680850,116621032,None
3,0x04916039b1f59d9745bf6e0a21f191d1e0a84287,14763568,39378616869897488346,37820764795752297020,180001972151225064917028985600
4,0x04a959e3b14bcb1e5c35c611ba5d4a1bf29ad26d,14763568,656157863,459265894,2264254457188625502368231966


In [145]:
# filter based on liquidity
mask = []
mask_filter = 10e12
current = prices[prices['block_number'] == prices['block_number'].max()]
current.reset_index()
for index, row in current.iterrows():
    k = int(row['reserve_t0'])*int(row['reserve_t1'])
    mask.append(k > mask_filter)
    
filtered = list(current[mask]['pool_address'])
filtered_pools = masked_pools[[adr in filtered for adr in masked_pools['address']]]
filtered_pools.to_csv('../data/pools_liq_degree_filter.csv')
